In [1]:
import getpass
import os
from dotenv import load_dotenv

load_dotenv('../.env')
openai_api_key = os.getenv('OPENAI_ACCESS_KEY')
huggingface_access_key = os.getenv("HUGGINGFACEHUB_API_TOKEN")

In [2]:
from langchain_elasticsearch import ElasticsearchStore
from langchain_community.embeddings import HuggingFaceEmbeddings

from typing import Any, Dict, Iterable

from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk
from langchain.embeddings import DeterministicFakeEmbedding
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_elasticsearch import ElasticsearchRetriever
from sentence_transformers import SentenceTransformer

/home/mabdallah/miniconda3/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

class SentenceEmbedder:
    def __init__(self, model_name: str = 'dmlls/all-mpnet-base-v2-negation'):
        # Load model and tokenizer from HuggingFace Hub
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)

    def mean_pooling(self, model_output, attention_mask):
        # Mean Pooling - Take attention mask into account for correct averaging
        token_embeddings = model_output[0]  # First element of model_output contains all token embeddings
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    def get_embeddings(self, sentence: str) -> list:
        # Tokenize sentence
        encoded_input = self.tokenizer([sentence], padding=True, truncation=True, return_tensors='pt')

        # Compute token embeddings
        with torch.no_grad():
            model_output = self.model(**encoded_input)

        # Perform pooling
        sentence_embeddings = self.mean_pooling(model_output, encoded_input['attention_mask'])

        # Normalize embeddings
        sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)

        # Convert to list and return
        return sentence_embeddings[0].tolist()


embedder = SentenceEmbedder()



/home/mabdallah/miniconda3/lib/python3.11/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [4]:
# Your Elasticsearch Cloud endpoint
es_url = "https://4616afc5fbda42dfa82407bdaf369e18.us-central1.gcp.cloud.es.io"

# Basic authentication details
username = "elastic"
password = "gWQzXpyXFfQp9q3tWgBwCGNG"

# Create an Elasticsearch client with API Key authentication
es_client = Elasticsearch(
    hosts=[es_url],
    http_auth=(username, password),
)

# Test the connection
info = es_client.info()
print(info)


/tmp/ipykernel_1523072/875203551.py:9: DeprecationWarning: The 'http_auth' parameter is deprecated. Use 'basic_auth' or 'bearer_auth' parameters instead
  es_client = Elasticsearch(


{'name': 'instance-0000000001', 'cluster_name': '4616afc5fbda42dfa82407bdaf369e18', 'cluster_uuid': 'M1TmpjezR3-smwcaMQr3ng', 'version': {'number': '8.13.4', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': 'da95df118650b55a500dcc181889ac35c6d8da7c', 'build_date': '2024-05-06T22:04:45.107454559Z', 'build_snapshot': False, 'lucene_version': '9.10.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


In [ ]:
from elasticsearch import Elasticsearch

def create_index(es_client: Elasticsearch, index_name: str, vector_dims: int):
    es_client.indices.create(
        index=index_name,
        body={
            "settings": {
                "analysis": {
                    "analyzer": {
                        "standard_lowercase": {
                            "type": "custom",
                            "tokenizer": "standard",
                            "filter": ["lowercase"]
                        }
                    }
                }
            },
            "mappings": {
                "properties": {
                    "criterion": {
                        "type": "text",
                        "analyzer": "standard_lowercase"
                    },
                    "criterion_vector": {  # Renamed and moved to be a separate top-level field
                        "type": "dense_vector",
                        "dims": vector_dims
                    },
                    "entities": {
                        "type": "nested",
                        "properties": {
                            "normalized_id": {"type": "keyword"},
                            "synonyms": {"type": "text", "analyzer": "standard_lowercase"},
                            "is_negated": {"type": "keyword"},
                            "entity": {"type": "text", "analyzer": "standard_lowercase"},
                            "class": {"type": "keyword"}
                        }
                    },
                    
                    "nct_id": {"type": "keyword", "index": False}  # Stored but not indexed
                }
            }
        },
    )


In [ ]:
import os
import json

def prepare_documents(folder_path):
    documents = []
    # Loop through all files in the specified directory
    for filename in os.listdir(folder_path):
        if filename.endswith('.json'):  # Check for JSON files
            file_path = os.path.join(folder_path, filename)
            with open(file_path, 'r') as file:
                data = json.load(file)  # Load JSON content
                nct_id = data['nct_id']  # Extract the NCT ID
                for criterion in data['criteria']:  # Iterate through each criterion
                    # Create a document for Elasticsearch
                    document = {
                        'criterion': criterion['criterion'],
                        'entities': criterion['entities'],
                        'eligibility_type': criterion['type'],
                        'nct_id': nct_id  # Include metadata containing NCT ID
                        
                    }
                    documents.append(document)
    return documents

# Example usage
folder_path = '../../data/ner_trial/'
documents = prepare_documents(folder_path)
print(f"Prepared {len(documents)} documents for indexing.")


In [ ]:
# Apply the function
number_of_indexed_documents = index_data(
    es_client=es_client,
    index_name="clinicaltrials",
    embedder=embedder,
    documents=documents,
    vector_dims=768,
)

In [5]:
def count_documents(es_client, index_name):
    try:
        count = es_client.count(index=index_name)
        print(f"Total documents in the index '{index_name}': {count['count']}")
        return count['count']
    except Exception as e:
        print(f"Error counting documents in index '{index_name}': {str(e)}")
        return None


In [6]:
count_documents(es_client, "clinicaltrials")

Total documents in the index 'clinicaltrials': 65021


65021

In [ ]:
def vector_query(search_query: str) -> Dict:
    vector = embedder.get_embeddings(search_query)  # same embeddings as for indexing
    return {
        "size": 500,  # return top 10 results
        "knn": {
            "field": "criterion_vector",
            "query_vector": vector,
            "num_candidates": 10000,
        }
    }

vector_retriever = ElasticsearchRetriever.from_es_params(
    index_name="clinicaltrials",
    body_func=vector_query,
    content_field="criterion",
    url=es_url,
    username=username,
    password=password,
)

vector_retriever.invoke("KRAS mutation")

In [31]:
text_field = "criterion"
index_name = "clinicaltrials"
from elasticsearch import Elasticsearch


def bm25_query(search_query: str, eligibility_type: str = "Inclusion Criteria") -> Dict:
    vector = embedder.get_embeddings(search_query)
    return {
        "size": 5000,  # Controls the number of search results returned
        "min_score": 50,  # Minimum BM25 score to consider
        "query": {
            "bool": {
                "must": [
                    {
                        "match": {
                            "eligibility_type.keyword": eligibility_type
                        }
                    },
                    {
                        "bool": {
                            "should": [
                                {
                                    "match": {
                                        "criterion": {
                                            "query": search_query,
                                            "fuzziness": "AUTO"
                                        }
                                    }
                                },
                                {
                                    "nested": {
                                        "path": "entities",
                                        "query": {
                                            "bool": {
                                                "should": [
                                                    {
                                                        "match": {
                                                            "entities.entity": {
                                                                "query": search_query,
                                                                "fuzziness": "AUTO"
                                                            }
                                                        }
                                                    },
                                                    {
                                                        "match": {
                                                            "entities.synonyms": {
                                                                "query": search_query,
                                                                "fuzziness": "AUTO"
                                                            }
                                                        }
                                                    }
                                                ]
                                            }
                                        }
                                    }
                                }
                            ]
                        }
                    }
                ]
            }
        },
    }


bm25_retriever = ElasticsearchRetriever.from_es_params(
    index_name=index_name,
    body_func=bm25_query,
    content_field=text_field,
    url=es_url,
    username=username,
    password=password,
)

# bm25_retriever.invoke("KRAS")

In [32]:
documents_result = bm25_retriever.invoke("Mutations in the KRAS gene are associated with poor prognosis in patients with non-small cell lung cancer.") 

In [50]:
documents_result[0].metadata['_source']["nct_id"]

'NCT02544633'

In [ ]:
from langchain.retrievers import EnsembleRetriever
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever], weights=[0.4, 0.6]
)

In [ ]:
docs = ensemble_retriever.invoke("KRAS mutation")
docs

In [ ]:
len(docs)

In [ ]:
text_field = "criterion"
index_name = "clinicaltrials"
def hybrid_query(search_query: str) -> Dict:
    vector = embeddings.embed_query(search_query)  # same embeddings as for indexing
    return {
        "size": 500,  # Controls the number of search results returned
                "query": {
            "match": {
               text_field: {
                    "query": search_query,
                    "fuzziness": "AUTO",
                }
            },
        },
        "knn": {
            "field": "criterion_vector",
            "query_vector": vector,
            "k": 500,
        },
        "rank": {"rrf": {"window_size": 1000}},
    }


hybrid_retriever = ElasticsearchRetriever.from_es_params(
    index_name=index_name,
    body_func=hybrid_query,
    content_field=text_field,
    url=es_url,
    username=username,
    password=password,
)

docs = hybrid_retriever.invoke("KRAS mutaion")


text_field = "criterion"
vector_retriever = ElasticsearchRetriever.from_es_params(
    index_name="clinicaltrials",
    body_func=vector_query,
    content_field=text_field,
    url=es_url,
    username=username,
    password=password,
)

vector_retriever.invoke("Retinal vein occlusion")

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

sentences = "The patient has a history of retinal vein occlusion.",
    # "The patient does have a history of retinal vein occlusion.",


model = SentenceTransformer(model_name_or_path="dmlls/all-mpnet-base-v2-negation")
len(model.encode(sentences).tolist()[0])

# def cosine_similarity(v1, v2):
#     return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

In [ ]:
embeddings[0].tolist()

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

class SentenceEmbedder:
    def __init__(self, model_name: str = 'dmlls/all-mpnet-base-v2-negation'):
        # Load model and tokenizer from HuggingFace Hub
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)

    def mean_pooling(self, model_output, attention_mask):
        # Mean Pooling - Take attention mask into account for correct averaging
        token_embeddings = model_output[0]  # First element of model_output contains all token embeddings
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    def get_embeddings(self, sentence: str) -> list:
        # Tokenize sentence
        encoded_input = self.tokenizer([sentence], padding=True, truncation=True, return_tensors='pt')

        # Compute token embeddings
        with torch.no_grad():
            model_output = self.model(**encoded_input)

        # Perform pooling
        sentence_embeddings = self.mean_pooling(model_output, encoded_input['attention_mask'])

        # Normalize embeddings
        sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)

        # Convert to list and return
        return sentence_embeddings[0].tolist()

# Example usage
sentence = documents[0]['criterion']
embedder = SentenceEmbedder()
embedding = embedder.get_embeddings(sentence)
print(embedding)


In [ ]:
embedding = embedder.get_embeddings("hello")



In [ ]:
documents[0]['criterion']

In [ ]:
print(embedding)